# Azure DevOps Wiki Helpers

Azure DevOps Wiki sayfalarini okuma/olusturma/guncelleme icin ortak fonksiyonlar.
PAT, Databricks Secret Scope uzerinden okunur.

In [0]:
import requests
import base64
from urllib.parse import quote


organization = "ozanozeer"
project = "aXet Project"
wiki = "aXet-Project.wiki"


pat = dbutils.secrets.get(
    scope="sql-ozoezer",
    key="devopspac"
)


auth = base64.b64encode(
    f":{pat}".encode()
).decode()


headers = {
    "Authorization": f"Basic {auth}",
    "Content-Type": "application/json"
}


def get_wiki_page(page_path):

    project_encoded = quote(project)
    page_path_encoded = quote(page_path, safe="/")

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path_encoded}"
        f"&api-version=7.1"
    )

    response = requests.get(
        url,
        headers=headers
    )

    return response


def create_wiki_page(page_path, content):

    project_encoded = quote(project)
    page_path_encoded = quote(page_path, safe="/")

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path_encoded}"
        f"&api-version=7.1"
    )

    response = requests.put(
        url,
        headers=headers,
        json={
            "content": content
        }
    )

    return response


def update_wiki_page(page_path, content, version):

    project_encoded = quote(project)
    page_path_encoded = quote(page_path, safe="/")

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path_encoded}"
        f"&api-version=7.1"
    )

    headers_update = headers.copy()

    headers_update["If-Match"] = version

    response = requests.put(
        url,
        headers=headers_update,
        json={
            "content": content
        }
    )

    return response


def push_wiki_page(page_path, content):

    existing_page = get_wiki_page(page_path)

    if existing_page.status_code == 200:

        # Azure DevOps ETag header'dan alınır
        version = existing_page.headers.get("ETag")

        if version is None:
            print("ETag bulunamadı")
            print(existing_page.headers)
            return existing_page

        response = update_wiki_page(
            page_path,
            content,
            version
        )

        print("Wiki page updated")

    else:

        response = create_wiki_page(
            page_path,
            content
        )

        print("Wiki page created")


    print(response.status_code)
    print(response.text)

    return response

# Web Fetch Helpers

Disaridan HTML/metin veri cekerken robots.txt kontrolu ve genel amacli
HTTP GET icin ortak fonksiyonlar.


In [0]:
from urllib.parse import urlparse
from urllib import robotparser


def check_robots_allowed(url, user_agent="*"):

    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

    try:
        robots_response = requests.get(robots_url, timeout=15)
    except Exception:
        return True

    if robots_response.status_code >= 400:
        return True

    parser = robotparser.RobotFileParser()
    parser.parse(robots_response.text.splitlines())

    return parser.can_fetch(user_agent, url)


def fetch_url_text(url, request_headers=None, timeout=30):

    response = requests.get(
        url,
        headers=request_headers,
        timeout=timeout,
    )
    response.raise_for_status()

    return response.text
